<a href="https://colab.research.google.com/github/sagarpanchal008/promptforge-ai/blob/main/Assignment_2_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Assignment 2: Individual Development of a Large Language Model (LLM) and Technical Presentation**

**Project: Clinical Symptom & Triage Assistant**

**Step 1: Install Required Libraries**

We install transformers and datasets to load the pretrained open-source LLM and manage data splits, bitsandbytes for 4-bit model quantization, peft to attach low-rank LoRA adapters, trl for supervised fine-tuning support, scikit-learn for evaluation metrics (accuracy_score, f1_score, classification_report), and gradio for a live interactive web demo.

In [2]:
# Install Hugging Face libraries for model loading, QLoRA/PEFT, datasets, training, evaluation, and UI
!pip install -q -U transformers peft accelerate bitsandbytes datasets trl scikit-learn gradio

**Step 2: Verify GPU & Colab T4 Environment**

In [3]:
import torch

# Check if a GPU is available in the Colab runtime
print("CUDA available:", torch.cuda.is_available())

# Print GPU name and total VRAM so we can confirm we are on a T4 (16 GB)
if torch.cuda.is_available():
  print("Device:", torch.cuda.get_device_name(0))
  print(
      "VRAM (GB):",
      round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2),
  )

CUDA available: True
Device: Tesla T4
VRAM (GB): 15.64


**Step 3: Create Balanced Training & Test Datasets**

We construct a balanced 150-example triage dataset (50 Emergency, 50 Urgent, 50 Routine) where each input maps to a consistent JSON object containing urgency, specialist, next_steps, and red_flags. Holding out 30 examples for testing lets us measure generalization on unseen phrasing without data leakage.


In [4]:
import json
import random
from datasets import Dataset

# Fix random seed so dataset shuffling and train/test splits are reproducible
random.seed(42)

# Define representative triage cases across three urgency tiers: Emergency, Urgent, and Routine
seed_cases = [
    # --- EMERGENCY CASES (life/limb-threatening signs requiring immediate emergency care) ---
    (
        "Crushing chest pain radiating to left arm and jaw for 45 minutes with"
        " cold sweat and shortness of breath.",
        "Emergency",
        "Cardiology / Emergency Medicine",
        "Call emergency services immediately; do not drive yourself.",
        ["Chest pain radiating to arm/jaw", "Shortness of breath", "Diaphoresis"],
    ),
    (
        "Sudden weakness on the right side of my face and slurred speech that"
        " started 20 minutes ago.",
        "Emergency",
        "Neurology / Emergency Medicine",
        "Seek emergency stroke care immediately; note exact symptom onset time.",
        ["Focal facial droop", "Speech slurring", "Sudden neurological deficit"],
    ),
    (
        "Severe difficulty breathing and throat swelling after eating peanuts ten"
        " minutes ago.",
        "Emergency",
        "Emergency Medicine / Allergy",
        "Call emergency services immediately; use epinephrine auto-injector if available.",
        ["Airway/throat swelling", "Acute breathing distress", "Possible anaphylaxis"],
    ),
    (
        "Coughing up bright red blood along with sharp chest pain and dizziness.",
        "Emergency",
        "Pulmonology / Emergency Medicine",
        "Go to the nearest emergency department immediately.",
        ["Hemoptysis", "Dizziness/presyncope", "Acute chest pain"],
    ),
    (
        "Sudden worst headache of my life with neck stiffness, fever, and light"
        " sensitivity.",
        "Emergency",
        "Neurology / Emergency Medicine",
        "Seek emergency medical evaluation immediately.",
        ["Thunderclap headache", "Neck stiffness", "Fever"],
    ),
    # --- URGENT CASES (prompt same-day or same-shift medical evaluation needed) ---
    (
        "High fever of 39.4°C for two days with painful urination and lower"
        " back flank pain.",
        "Urgent",
        "Urology / Internal Medicine",
        "See an urgent care clinic or doctor today for urine testing and evaluation.",
        ["High fever", "Flank pain", "Possible upper UTI/pyelonephritis"],
    ),
    (
        "Deep cut on my forearm from kitchen glass that stopped bleeding after"
        " firm pressure but may need stitches.",
        "Urgent",
        "Urgent Care / General Surgery",
        "Visit urgent care within a few hours for wound cleaning and closure check.",
        ["Deep laceration", "Tetanus/infection risk", "Nerve/tendon check needed"],
    ),
    (
        "Persistent vomiting and inability to keep fluids down for 24 hours with"
        " lightheadedness when standing.",
        "Urgent",
        "Internal Medicine / Urgent Care",
        "Seek same-day urgent evaluation for hydration and electrolyte assessment.",
        ["24h fluid intolerance", "Orthostatic lightheadedness", "Dehydration risk"],
    ),
    (
        "Painful, swollen red right calf that started two days ago after a long"
        " flight.",
        "Urgent",
        "Vascular Medicine / Internal Medicine",
        "Get same-day medical evaluation to rule out deep vein thrombosis.",
        ["Unilateral calf swelling/pain", "Recent immobility/travel", "DVT risk"],
    ),
    (
        "Ear pain for three days with fever of 38.5°C and fluid drainage from"
        " the ear.",
        "Urgent",
        "ENT (Otolaryngology) / Primary Care",
        "Schedule a same-day or next-day doctor visit for ear examination.",
        ["Fever with otorrhea", "Worsening ear pain", "Possible acute otitis media"],
    ),
    # --- ROUTINE CASES (mild or non-acute symptoms suitable for scheduled/routine care) ---
    (
        "Mild scratchy throat and clear runny nose for two days without fever or"
        " breathing trouble.",
        "Routine",
        "Primary Care / General Medicine",
        "Rest, stay hydrated, use saline/warm gargles, and book a routine visit if persistent.",
        ["No high fever", "No shortness of breath", "Mild upper respiratory symptoms"],
    ),
    (
        "Occasional mild lower back stiffness after sitting at a desk all day,"
        " improving with stretching.",
        "Routine",
        "Primary Care / Orthopedics",
        "Maintain gentle movement and posture breaks; book a routine appointment if ongoing.",
        ["No numbness or leg weakness", "No bowel/bladder changes", "Mechanical strain"],
    ),
    (
        "Dry, itchy patches of skin on my elbows for two weeks without pus,"
        " warmth, or fever.",
        "Routine",
        "Dermatology / Primary Care",
        "Use fragrance-free moisturizer and schedule a routine dermatology or PCP visit.",
        ["No signs of spreading infection", "Chronic mild rash", "No systemic symptoms"],
    ),
    (
        "Mild seasonal sneezing and watery eyes every spring when pollen is high.",
        "Routine",
        "Allergy & Immunology / Primary Care",
        "Consider standard allergy precautions and schedule a routine consultation.",
        ["No wheezing or chest tightness", "Seasonal pattern", "Mild symptoms"],
    ),
    (
        "Looking to schedule a routine annual wellness checkup and basic blood"
        " workup; feeling well overall.",
        "Routine",
        "Primary Care / Preventive Medicine",
        "Book a standard preventive care appointment at your convenience.",
        ["Asymptomatic", "Preventive screening", "No acute complaints"],
    ),
]

# Intake prefixes and suffixes to vary surface phrasing while preserving ground-truth triage labels
prefixes = [
    "Patient reports: ",
    "Symptoms for evaluation: ",
    "Chief complaint: ",
    "User note: ",
    "Clinical intake summary: ",
]
suffixes = [
    "",
    " No known drug allergies reported.",
    " Symptoms are currently ongoing.",
    " Asking for guidance on next steps.",
    " Please triage carefully.",
]

all_records = []
for symp, urg, spec, steps, flags in seed_cases:
  # Generate 10 phrasing variations per seed case -> 150 total records (50 per class)
  for idx in range(10):
    p = prefixes[idx % len(prefixes)]
    s = suffixes[(idx // len(prefixes)) % len(suffixes)]
    # Serialize target output as a structured JSON string
    target_json = json.dumps(
        {
            "urgency": urg,
            "specialist": spec,
            "next_steps": steps,
            "red_flags": flags,
        },
        ensure_ascii=False,
    )
    all_records.append(
        {"symptoms": f"{p}{symp}{s}".strip(), "target": target_json}
    )

# Shuffle records and split into 120 training examples and 30 held-out test examples
random.shuffle(all_records)
test_records = all_records[:30]
train_records = all_records[30:]

# Convert Python lists into Hugging Face Dataset objects
train_ds = Dataset.from_list(train_records)
test_ds = Dataset.from_list(test_records)
print(f"Train size: {len(train_ds)} | Test size: {len(test_ds)}")

Train size: 120 | Test size: 30


**Step 4: Load Qwen/Qwen2.5-1.5B-Instruct & Fine-Tune with 4-bit QLoRA**

We load Qwen/Qwen2.5-1.5B-Instruct in 4-bit NF4 with float16 compute, wrap its attention and feed-forward linear layers with LoRA (r=16, lora_alpha=32), format examples with apply_chat_template, and train for 3 epochs with an effective batch size of 16. Because the base weights stay frozen in 4-bit, only a small fraction of adapter parameters are updated, keeping memory usage low and training fast.

In [5]:
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

# Model identifier on Hugging Face Hub (1.54B parameters, instruction-tuned)
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# System instruction enforcing strict JSON output schema and educational scope
SYSTEM_PROMPT = (
    "You are a clinical triage assistant. Given patient symptoms, respond ONLY"
    " with valid JSON containing keys: urgency"
    " ('Emergency'|'Urgent'|'Routine'), specialist, next_steps, and red_flags"
    " (list of strings). This is for educational triage support, not medical"
    " diagnosis."
)

# Configure 4-bit NormalFloat (NF4) quantization with float16 compute dtype for T4 compatibility
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer and ensure a pad token is defined for batch padding
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

# Load the pretrained causal language model quantized to 4-bit across available GPU
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
# Disable KV cache during training (required when using gradient checkpointing/k-bit training)
base_model.config.use_cache = False


def format_prompt(symptoms, target=None):
  """Format conversation messages using the model's native chat template."""
  messages = [
      {"role": "system", "content": SYSTEM_PROMPT},
      {"role": "user", "content": symptoms},
  ]
  # Include assistant response during training; omit during inference and add generation prompt
  if target is not None:
    messages.append({"role": "assistant", "content": target})
  return tokenizer.apply_chat_template(
      messages, tokenize=False, add_generation_prompt=(target is None)
  )


def tokenize_example(row):
  """Convert a formatted text sample into fixed-length token IDs and causal LM labels."""
  full_text = format_prompt(row["symptoms"], row["target"])
  enc = tokenizer(
      full_text, truncation=True, max_length=384, padding="max_length"
  )
  # For causal language modeling, labels match input_ids so the model learns next-token prediction
  enc["labels"] = enc["input_ids"].copy()
  return enc


# Tokenize the training dataset and drop raw string columns
tokenized_train = train_ds.map(
    tokenize_example, remove_columns=train_ds.column_names
)

# Prepare quantized model for k-bit training and attach LoRA adapters
model = prepare_model_for_kbit_training(base_model)
lora_config = LoraConfig(
    r=16,  # Rank of low-rank update matrices
    lora_alpha=32,  # Scaling factor (2 * r is a common default)
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],  # Target attention and MLP linear projections
    lora_dropout=0.05,  # Dropout for regularization
    bias="none",  # Do not train bias parameters
    task_type="CAUSAL_LM",  # Causal language modeling task
)
model = get_peft_model(model, lora_config)

# Display how many parameters are trainable vs. frozen in the base model
model.print_trainable_parameters()

# Configure training hyperparameters (fp16 is used instead of bf16 on Colab T4)
training_args = TrainingArguments(
    output_dir="./qwen25_triage_lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch size = 4 * 4 = 16
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
)

# Initialize Trainer with a causal LM data collator (mlm=False)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

# Run supervised fine-tuning on the LoRA adapter weights
trainer.train()

# Save only the trained LoRA adapter weights and tokenizer locally
model.save_pretrained("./qwen25_triage_adapter")
tokenizer.save_pretrained("./qwen25_triage_adapter")
print("Training complete and adapter saved.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

... (3 outputs left out)

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


Step,Training Loss
10,1.550366
20,0.453779


Training complete and adapter saved.


**Step 5: Evaluate Before vs. After Fine-Tuning on the Test Set**

We benchmark the un-tuned base model (disable_adapter()) against our QLoRA fine-tuned model on the same 30 unseen test examples. Reporting Valid JSON Rate, Urgency Accuracy, and Macro-F1 captures both structured schema reliability and balanced performance across all three triage classes.

In [6]:
import json
import re
from sklearn.metrics import accuracy_score, classification_report, f1_score


def generate_triage(symptoms_text, use_adapter=True):
  """Generate a triage response from either the base model (use_adapter=False) or fine-tuned model."""
  model.eval()
  prompt_text = format_prompt(symptoms_text, target=None)
  inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

  # Run deterministic greedy decoding (do_sample=False) for consistent evaluation
  with torch.no_grad():
    if not use_adapter:
      # Temporarily disable LoRA weights to measure the un-tuned base model baseline
      with model.disable_adapter():
        out = model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    else:
      out = model.generate(
          **inputs,
          max_new_tokens=220,
          do_sample=False,
          pad_token_id=tokenizer.pad_token_id,
      )

  # Slice off input prompt tokens so we decode only newly generated assistant tokens
  gen_ids = out[0][inputs["input_ids"].shape[-1] :]
  raw = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
  return raw


def parse_json_safe(raw_str):
  """Extract and parse the first JSON object from model output, returning None if invalid."""
  match = re.search(r"\{.*\}", raw_str, flags=re.DOTALL)
  if not match:
    return None
  try:
    return json.loads(match.group(0))
  except json.JSONDecodeError:
    return None


# Compare the base model against the QLoRA fine-tuned model on all 30 held-out test cases
for mode_name, flag in [("Base Model (Before QLoRA)", False), (
    "Fine-Tuned Model (After QLoRA)",
    True,
)]:
  valid_json = 0
  y_true, y_pred = [], []

  for row in test_ds:
    gt = json.loads(row["target"])
    raw = generate_triage(row["symptoms"], use_adapter=flag)
    parsed = parse_json_safe(raw)

    y_true.append(gt["urgency"])
    if parsed and isinstance(parsed, dict) and "urgency" in parsed:
      valid_json += 1
      y_pred.append(str(parsed["urgency"]))
    else:
      # Mark unparseable or schema-missing outputs as "Invalid"
      y_pred.append("Invalid")

  acc = accuracy_score(y_true, y_pred)
  mf1 = f1_score(
      y_true,
      y_pred,
      labels=["Emergency", "Urgent", "Routine"],
      average="macro",
      zero_division=0,
  )

  print(f"\n=== {mode_name} ===")
  print(f"Valid JSON Rate: {valid_json}/{len(test_ds)}")
  print(f"Urgency Accuracy: {acc:.2%} | Macro-F1: {mf1:.2%}")
  print(
      classification_report(
          y_true,
          y_pred,
          labels=["Emergency", "Urgent", "Routine"],
          zero_division=0,
      )
  )


=== Base Model (Before QLoRA) ===
Valid JSON Rate: 30/30
Urgency Accuracy: 83.33% | Macro-F1: 83.83%
              precision    recall  f1-score   support

   Emergency       0.88      0.88      0.88         8
      Urgent       0.69      0.90      0.78        10
     Routine       1.00      0.75      0.86        12

    accuracy                           0.83        30
   macro avg       0.86      0.84      0.84        30
weighted avg       0.86      0.83      0.84        30


=== Fine-Tuned Model (After QLoRA) ===
Valid JSON Rate: 30/30
Urgency Accuracy: 90.00% | Macro-F1: 90.25%
              precision    recall  f1-score   support

   Emergency       0.89      1.00      0.94         8
      Urgent       0.82      0.90      0.86        10
     Routine       1.00      0.83      0.91        12

    accuracy                           0.90        30
   macro avg       0.90      0.91      0.90        30
weighted avg       0.91      0.90      0.90        30



**Step 6: Launch an Interactive Gradio Demo**

We expose the fine-tuned triage model through a Gradio web interface inside Colab so you can enter sample symptom descriptions live during your presentation and viva and show structured JSON responses immediately.

In [7]:
import gradio as gr


def predict_triage(symptoms):
  """Run the fine-tuned triage model on user input and pretty-print JSON when valid."""
  if not symptoms.strip():
    return "Please enter patient symptoms."
  raw = generate_triage(symptoms.strip(), use_adapter=True)
  parsed = parse_json_safe(raw)
  if parsed:
    return json.dumps(parsed, indent=2)
  # Fallback to raw text if the model returned non-JSON text
  return raw


# Build a simple web interface for live presentation and viva demonstration
demo = gr.Interface(
    fn=predict_triage,
    inputs=gr.Textbox(
        lines=4,
        label="Patient Symptoms",
        placeholder=(
            "Example: Sudden chest tightness and shortness of breath for 30"
            " minutes..."
        ),
    ),
    outputs=gr.Code(language="json", label="Structured Triage Output"),
    title="Clinical Symptom & Triage Assistant (Qwen2.5-1.5B-Instruct + QLoRA)",
    description=(
        "Educational NLP prototype adapting an open-source LLM with 4-bit"
        " QLoRA. Not a substitute for licensed medical care."
    ),
)

# Launch the Gradio app inside Colab
demo.launch(share=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>